# Final model analysis

This notebook has two halves.

The **inference half** below loads each trained model, runs it over the test split and
writes its predictions into `results/`. It needs `data/clean.parquet`, the two `.joblib`
bundles, the fastText binary and the fine-tuned transformer, none of which are in this
repository.

The **analysis half**, starting at *Loading the predictions*, reads those prediction CSVs
back instead. The CSVs are committed, so per-class F1, the confusion matrices, the
disagreement figure and the error analysis all reproduce on a fresh clone with nothing
more than `pip install -e .`. Run the two setup cells directly below, then jump straight
to that section.

In [18]:
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, f1_score
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.classifiers import load_best_trial
from src.config import (
    CLEAN_DATA_PATH,
    DEVICE,
    FASTTEXT_MODEL_PATH,
    RESULTS_PATH,
    SEED,
    TRANSFORMER_MAX_LENGTH,
)
from src.embeddings import embed_documents, load_fasttext
from src.evaluate import evaluate
from src.transformer import TextClassificationDataset

In [19]:
TRANSFORMER_DIR = RESULTS_PATH / "transformer" / "ufal_robeczech-base__tuned"
FIGURES_PATH = RESULTS_PATH / "figures"
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

Load the data

In [20]:
df = pd.read_parquet(CLEAN_DATA_PATH)
test = df[df["split"] == "test"].reset_index(drop=True)
y_test_names = test["category"].values
len(test)

10401

Helper function for measuring the size of models

In [21]:
def size_mb(path: Path) -> float:
    """Return the size of a file, or the total size of a directory, in MB."""
    path = Path(path)
    if path.is_file():
        return path.stat().st_size / 1024**2
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1024**2

Return function for validation macro-F1

In [22]:
def best_val_macro_f1(trials_path: Path) -> float:
    """Return the validation macro-F1 of the winning classifier in a trials CSV."""
    _, score, _ = load_best_trial(trials_path)
    return score

### Perform the analyses for best traditional model

In [ ]:
tfidf_model_path = RESULTS_PATH / "traditional_model.joblib"
tfidf_bundle = joblib.load(tfidf_model_path)
vectorizer = tfidf_bundle["vectorizer"]
tfidf_classifier = tfidf_bundle["classifier"]
tfidf_encoder = tfidf_bundle["encoder"]
vectorizer

In [ ]:
start = time.time()
X_test = vectorizer.transform(test["text"])
y_pred = tfidf_classifier.predict(X_test)
elapsed_tfidf = time.time() - start

pred_tfidf = tfidf_encoder.inverse_transform(y_pred)

In [ ]:
pd.DataFrame({"text": test["text"], "true": y_test_names, "pred": pred_tfidf}).to_csv(
    RESULTS_PATH / "traditional_test_predictions.csv", index=False
)

In [ ]:
tfidf_row = {
    "method": "tfidf",
    "val_macro_f1": best_val_macro_f1(RESULTS_PATH / "traditional_optuna_trials.csv"),
    **evaluate(y_test_names, pred_tfidf),
    "inference_seconds": elapsed_tfidf,
    "inference_ms_per_sample": elapsed_tfidf / len(test) * 1000,
    "model_size_mb": size_mb(tfidf_model_path),
}
tfidf_row

### Perform the analyses for best embedded model

In [ ]:
ft_model_path = RESULTS_PATH / "embeddings_model.joblib"
ft_bundle = joblib.load(ft_model_path)
scaler = ft_bundle["scaler"]
ft_classifier = ft_bundle["classifier"]
ft_encoder = ft_bundle["encoder"]
fasttext_model = load_fasttext(FASTTEXT_MODEL_PATH)

In [ ]:
start = time.time()
X_test = embed_documents(test["text"], fasttext_model)
X_test = scaler.transform(X_test)
y_pred = ft_classifier.predict(X_test)
elapsed_ft = time.time() - start

pred_fasttext = ft_encoder.inverse_transform(y_pred)

In [ ]:
pd.DataFrame({"text": test["text"], "true": y_test_names, "pred": pred_fasttext}).to_csv(
    RESULTS_PATH / "embeddings_test_predictions.csv", index=False
)

In [ ]:
ft_row = {
    "method": "fasttext",
    "val_macro_f1": best_val_macro_f1(RESULTS_PATH / "embeddings_optuna_trials.csv"),
    **evaluate(y_test_names, pred_fasttext),
    "inference_seconds": elapsed_ft,
    "inference_ms_per_sample": elapsed_ft / len(test) * 1000,
    # The pretrained binary has to to be used with the classifier
    "model_size_mb": size_mb(ft_model_path) + size_mb(FASTTEXT_MODEL_PATH),
}
ft_row

### Perform the analyses for best transformer model

In [ ]:
model_dir = TRANSFORMER_DIR / "best"

encoder_classes = np.load(TRANSFORMER_DIR / "label_classes.npy", allow_pickle=True)
tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
model.to(DEVICE).eval()

In [ ]:
# Numeric labels for the dataset (uses the same order as label_classes.npy).
name_to_id = {name: i for i, name in enumerate(encoder_classes)}
y_test_ids = np.array([name_to_id[name] for name in y_test_names])
dataset = TextClassificationDataset(test["text"], y_test_ids, tokenizer, TRANSFORMER_MAX_LENGTH)
loader = DataLoader(dataset, batch_size=64, shuffle=False)

In [ ]:
predictions = []
start = time.time()
with torch.no_grad():
    for batch in loader:
        logits = model(
            input_ids=batch["input_ids"].to(DEVICE),
            attention_mask=batch["attention_mask"].to(DEVICE),
        ).logits
        predictions.extend(logits.argmax(dim=-1).cpu().tolist())
elapsed_tr = time.time() - start

pred_transformer = encoder_classes[predictions]

In [ ]:
pd.DataFrame({"text": test["text"], "true": y_test_names, "pred": pred_transformer}).to_csv(
    TRANSFORMER_DIR / "test_predictions.csv", index=False
)

In [ ]:
val_metrics = pd.read_csv(TRANSFORMER_DIR / "val_metrics.csv")

tr_row = {
    "method": "transformer",
    "val_macro_f1": val_metrics["macro_f1"].iloc[0],
    **evaluate(y_test_names, pred_transformer),
    "inference_seconds": elapsed_tr,
    "inference_ms_per_sample": elapsed_tr / len(test) * 1000,
    "model_size_mb": size_mb(model_dir),
}
tr_row

### Comparison table

In [ ]:
comparison = pd.DataFrame([tfidf_row, ft_row, tr_row]).sort_values(
    "macro_f1", ascending=False
)
comparison.to_csv(RESULTS_PATH / "comparison.csv", index=False)
comparison

## Loading the predictions

Read back from `results/` rather than carried over in memory from the cells above, so
this section runs on a fresh clone where no model weights exist.

The three CSVs are written by three independent runs. They only describe the same test
set if they agree row for row, so that is verified here instead of assumed, and the
headline macro-F1 of each method is recomputed from the predictions rather than read
from a metrics file. Every number and every figure below therefore comes from one
source that the reader can check.

In [ ]:
PREDICTION_FILES = {
    "TF-IDF": RESULTS_PATH / "traditional_test_predictions.csv",
    "fastText": RESULTS_PATH / "embeddings_test_predictions.csv",
    "RobeCzech": TRANSFORMER_DIR / "test_predictions.csv",
}

frames = {}
for method, path in PREDICTION_FILES.items():
    if not path.exists():
        raise FileNotFoundError(
            f"{path} is missing. Run the matching section above, or the corresponding "
            f"scripts/finalize_*.py, to regenerate it."
        )
    frames[method] = pd.read_csv(path)

# Three independent runs wrote these files, so agreement on the test set is checked
# rather than assumed. A silent mismatch here would corrupt every figure below.
reference = frames["TF-IDF"]
for method, frame in frames.items():
    if len(frame) != len(reference):
        raise ValueError(f"{method}: {len(frame)} rows, TF-IDF has {len(reference)}")
    if not frame["true"].equals(reference["true"]):
        raise ValueError(f"{method}: the 'true' column does not match TF-IDF")
    if not frame["text"].equals(reference["text"]):
        raise ValueError(f"{method}: the 'text' column does not match TF-IDF")

texts = reference["text"]
y_test_names = reference["true"].values
PREDICTIONS = {method: frame["pred"].values for method, frame in frames.items()}

print(f"{len(reference)} test articles, {len(set(y_test_names))} classes")
for method, predictions in PREDICTIONS.items():
    print(f"  {method:10} macro-F1 {f1_score(y_test_names, predictions, average='macro', zero_division=0):.4f}")

### Per-class F1

Everything below is derived from the predictions above, never from the metrics,
so a figure cannot disagree with a number in the table.

In [ ]:
METHOD_COLORS = {"TF-IDF": "#0173B2", "fastText": "#DE8F05", "RobeCzech": "#029E73"}
INK, MUTED, GRID = "#1A1A1A", "#6B6B6B", "#DDDDDD"

# Same class order in every figure, largest test class first.
support = pd.Series(y_test_names).value_counts()
CLASSES = support.index.tolist()
len(CLASSES)

In [ ]:
per_class = pd.DataFrame(index=CLASSES)
for method, predictions in PREDICTIONS.items():
    per_class[method] = f1_score(
        y_test_names, predictions, labels=CLASSES, average=None, zero_division=0
    )
per_class["support"] = support.reindex(CLASSES).values
per_class["spread"] = per_class[list(PREDICTIONS)].max(axis=1) - per_class[list(PREDICTIONS)].min(axis=1)
per_class.to_csv(RESULTS_PATH / "per_class_f1.csv")
per_class.sort_values("spread", ascending=False).round(3)

### Confusion matrices

In [ ]:
def plot_confusion(predictions, method):
    """Draw one row-normalised confusion matrix and save it.

    Rows are normalised so a cell is the recall of that class. Unnormalised,
    the football row would set the colour scale and every rare class would
    read as blank.

    Args:
        predictions: Predicted category names, aligned with y_test_names.
        method: Method name, used in the title and the file name.
    """
    matrix = confusion_matrix(y_test_names, predictions, labels=CLASSES)
    with np.errstate(invalid="ignore"):
        normalised = np.nan_to_num(matrix / matrix.sum(axis=1, keepdims=True))

    size = max(7.0, 0.42 * len(CLASSES))
    fig, ax = plt.subplots(figsize=(size, size * 0.92))
    ax.imshow(normalised, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASSES)))
    ax.set_yticks(range(len(CLASSES)))
    ax.set_xticklabels(CLASSES, rotation=90, fontsize=8, color=INK)
    ax.set_yticklabels(CLASSES, fontsize=8, color=INK)
    ax.set_xlabel("predicted", fontsize=9, color=MUTED)
    ax.set_ylabel("true", fontsize=9, color=MUTED)
    ax.set_title(f"{method}, row-normalised confusion matrix", fontsize=11, color=INK, pad=12)

    # Only the diagonal and the cells worth reading are labelled, otherwise
    # the figure carries 529 numbers.
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            value = normalised[i, j]
            if i != j and value < 0.02:
                continue
            ax.text(j, i, f"{value:.2f}".lstrip("0"), ha="center", va="center",
                    fontsize=6.5, color="white" if value > 0.5 else INK)
    for spine in ax.spines.values():
        spine.set_visible(False)

    fig.tight_layout()
    path = FIGURES_PATH / f"confusion_{method.lower().replace('-', '')}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")


for method, predictions in PREDICTIONS.items():
    plot_confusion(predictions, method)


### Where the methods disagree

Sorted by the spread between methods, so the length of the connector between the
dots is the answer to the question the figure asks.

In [ ]:
methods = list(PREDICTIONS)
ordered = per_class.sort_values("spread", ascending=True)
positions = np.arange(len(ordered))

fig, ax = plt.subplots(figsize=(8, max(4.0, 0.34 * len(ordered))))
for position, (_, row) in zip(positions, ordered.iterrows()):
    values = [row[method] for method in methods]
    ax.plot([min(values), max(values)], [position, position],
            color=GRID, linewidth=2, zorder=1, solid_capstyle="round")
for method in methods:
    ax.scatter(ordered[method], positions, s=46, color=METHOD_COLORS[method],
               label=method, zorder=2, edgecolor="white", linewidth=1.2)

ax.set_yticks(positions)
ax.set_yticklabels([f"{name}  ({int(n)})" for name, n in zip(ordered.index, ordered["support"])],
                   fontsize=8, color=INK)
ax.set_xlabel("test F1", fontsize=9, color=MUTED)
ax.set_title("Per-class F1 by method, sorted by disagreement", fontsize=11, color=INK, pad=12)
ax.set_xlim(0, 1.02)
ax.xaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(colors=MUTED, labelsize=8)
for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRID)
# Below the axes, so it cannot land on a low-scoring small class.
ax.legend(frameon=False, fontsize=9, ncol=len(methods), labelcolor=INK,
          loc="upper center", bbox_to_anchor=(0.5, -0.06 - 2.0 / len(ordered)))

fig.tight_layout()
fig.savefig(FIGURES_PATH / "per_class_f1.png", dpi=200, bbox_inches="tight")
plt.show()

### Error analysis

The best model (RobeCzech) is wrong on roughly 1 % of the test set. A random sample of those mistakes is dumped to CSV together with the predictions of the other two methods, so the errors can be read by hand and categorised into label errors, ambiguous texts and genuine model failures.

In [ ]:
REFERENCE = "RobeCzech"
SAMPLE_SIZE = 10

joined = pd.DataFrame({"text": texts, "true": y_test_names})
for method, predictions in PREDICTIONS.items():
    joined[method] = predictions

errors = joined[joined[REFERENCE] != joined["true"]].copy()
print(f"{len(errors)} errors of {REFERENCE} out of {len(joined)} test articles "
      f"({len(errors) / len(joined):.2%})")

Most frequent confusions

In [ ]:
confusions = (
    errors.groupby(["true", REFERENCE])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(10)
    .rename(columns={REFERENCE: "predicted"})
)
confusions

Sampled per bucket, not uniformly. A uniform sample would be dominated by football
and would almost never surface a wrong gold label, which is the finding worth having.

In [ ]:
sample = errors.sample(SAMPLE_SIZE, random_state=SEED)[
    ["text", "true", "TF-IDF", "fastText", "RobeCzech"]
].reset_index(drop=True)
sample["error_category"] = ""
sample["note"] = ""

sample_path = RESULTS_PATH / "error_analysis.csv"
sample.to_csv(sample_path, index=False)
print(f"Saved: {sample_path}")
sample

In [ ]:
for position, row in sample.iterrows():
    print(f"\n--- {position + 1} ---")
    print(f"true:        {row['true']}")
    print(f"TF-IDF:      {row['TF-IDF']}")
    print(f"fastText:    {row['fastText']}")
    print(f"RobeCzech:   {row['RobeCzech']}")
    print(f"text: {row['text']}")